In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit


In [2]:
# Chargement
df = pd.read_csv("../../data/processed/02_dataset_clean.csv")

# Date
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

df.shape


(436, 38)

In [3]:
TARGET = "taux_chomage_insee"

# Exclusion des autres taux de chômage (fuite d'information)
leakage_cols = [
    col for col in df.columns
    if "taux_chomage_" in col and col != TARGET
]

X = df.drop(columns=[TARGET, "date"] + leakage_cols)
y = df[TARGET]

X.shape


(436, 29)

In [4]:
X_imp = X.fillna(X.median())
y_imp = y.fillna(y.median())

mi = mutual_info_regression(X_imp, y_imp, random_state=42)

mi_rank = (
    pd.Series(mi, index=X.columns)
    .sort_values(ascending=False)
)

mi_rank


population_active                       3.052692
annee                                   3.037835
pib                                     2.519930
ict                                     2.301460
demandeur_femme_abcd_plus50             2.209882
demandeur_total_abcd_plus50             1.988123
demandeur_homme_abcd_plus50             1.837096
taux_euribor_3m                         1.783686
demandeur_total_abcd_total              1.720634
demandeur_homme_abcd_total              1.682992
demandeur_femme_abcd_total              1.615041
demandeur_homme_abcd_2549               1.573862
demandeur_total_abcd_2549               1.547677
MRO                                     1.509086
demandeur_femme_abcd_2549               1.508678
demandeur_total_abcd_moins25            1.452990
demandeur_femme_abcd_moins25            1.417896
demandeur_homme_abcd_moins25            1.199674
nb_interimaires                         0.952037
isj                                     0.897510
ipc                 

In [5]:
tscv = TimeSeriesSplit(n_splits=5)

lasso_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(
        cv=tscv,
        random_state=42,
        max_iter=20000
    ))
])

lasso_pipeline.fit(X, y)

coef = lasso_pipeline.named_steps["lasso"].coef_

lasso_rank = (
    pd.Series(coef, index=X.columns)
    .sort_values(key=abs, ascending=False)
)

lasso_rank


demandeur_femme_abcd_moins25            0.445479
MRO                                    -0.208959
nb_interimaires                        -0.167928
annee                                  -0.160541
isj                                    -0.101731
nb_defaillances_entreprise              0.095215
demandeur_femme_abcd_plus50            -0.088700
indicateur_retournement_conjoncturel    0.041823
nb_offres_france_travail               -0.024666
ict                                    -0.022304
demandeur_total_abcd_moins25            0.007613
taux_euribor_3m                         0.006519
mois                                    0.000000
pib                                    -0.000000
indicateur_climat_emploi                0.000000
indicateur_climat_affaires              0.000000
ipc_energie_only                        0.000000
ipc                                    -0.000000
demandeur_homme_abcd_total             -0.000000
demandeur_femme_abcd_total             -0.000000
trimestre           